<a href="https://colab.research.google.com/github/kanyasornp69/Teach-Urban-RS/blob/main/02_Urban_Vegetation_and_Builtup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — Vegetation, NDVI และ Built-up Area

**หัวข้อ:** Vegetation, NDVI และ Built-up Area

**คำถามหลัก:** *พื้นที่เมืองประกอบด้วย vegetation และ built-up surface ในรูปแบบและสัดส่วนอย่างไร?*

### วัตถุประสงค์การเรียนรู้
- แยกความแตกต่างระหว่าง **Land Cover** และ **Land Use**
- อธิบายหลักการพื้นฐานของ vegetation spectral response
- คำนวณและแปลความหมาย NDVI
- ใช้ Dynamic World `built` probability
- คำนวณพื้นที่ built-up จาก pixel area
- สร้าง thematic map ที่มี colorbar/legend และ boundary
- ระมัดระวังเรื่อง threshold และ study-area effect

## 1. แนวคิด: Land Cover ≠ Land Use

**Land Cover** = สิ่งที่ปกคลุมพื้นผิว เช่น built surface, trees, water, crops  
**Land Use** = มนุษย์ใช้พื้นที่ทำอะไร เช่น residential, commercial, industrial, park

Remote Sensing สังเกต **Land Cover** โดยตรงกว่า Land Use

## 2. NDVI

$$
NDVI = \frac{NIR - Red}{NIR + Red}
$$

สำหรับ Sentinel-2:
- NIR = B8
- Red = B4

Healthy vegetation มักดูดกลืนแสงช่วง Red และสะท้อน NIR สูง จึงให้ NDVI เป็นบวกค่อนข้างสูง

> **ข้อควรระวัง:** ไม่ควรใช้ threshold NDVI แบบตายตัวเพื่อระบุชนิด vegetation ทุกพื้นที่ เพราะค่าขึ้นกับ sensor, season, atmosphere, canopy และ land cover

## การเตรียมระบบ (ทำทุก Notebook)

Notebook นี้ออกแบบสำหรับ **Google Colab + Google Earth Engine Python API + geemap**

### ก่อนเริ่ม
1. ต้องมีบัญชี Google Earth Engine ที่เปิดใช้งานแล้ว
2. ต้องมี Google Cloud Project ที่ใช้กับ Earth Engine
3. แก้ค่า `PROJECT` ใน code cell ด้านล่างให้เป็น Project ID ของตนเอง

> **แนวคิดสำคัญ:** Python ใน Colab เป็นฝั่งผู้ใช้ (client) ส่วน `ee.Image`, `ee.ImageCollection` และการคำนวณส่วนใหญ่เป็นวัตถุแบบ server-side ที่ประมวลผลบน Google Earth Engine

In [3]:
%pip install -q earthengine-api geemap pandas matplotlib scikit-learn scipy

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Google Cloud Project ID ที่ลงทะเบียน Earth Engine แล้ว
PROJECT = "x-vector-510703-r9" #<- แก้ ด้วยการใช้ project ID ของนิสิต

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully.")
print("Using project:", PROJECT)

# -------------------------------------------------------------------
# ฟังก์ชันมาตรฐานสำหรับทุกแผนที่ในรายวิชา
# 1) สร้าง geemap.Map()
# 2) ลบ default OpenStreetMap ที่ถูก block ใน Colab environment นี้
# 3) ใช้ Esri.WorldImagery เป็น basemap เริ่มต้น
# 4) เตรียม Street Map และ Topographic Map ไว้ให้เลือกใน Layer Control
# -------------------------------------------------------------------
def create_course_map():
    m = geemap.Map()
    m.clear_layers()
    m.add_basemap("Esri.WorldImagery", show=True)
    m.add_basemap("Esri.WorldStreetMap", show=False)
    m.add_basemap("Esri.WorldTopoMap", show=False)
    return m

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized successfully.
Using project: x-vector-510703-r9


### Basemap ที่ใช้ในชุด Notebook นี้

ทุกแผนที่เรียกผ่าน `create_course_map()` เพื่อหลีกเลี่ยง default OpenStreetMap tile ที่ถูก block ใน Colab environment นี้ โดยกำหนด **Esri.WorldImagery** เป็นพื้นหลังเริ่มต้น และมี **Esri.WorldStreetMap** กับ **Esri.WorldTopoMap** ให้เปิดจาก Layer Control ตามวัตถุประสงค์ของการอ่านแผนที่


In [4]:
# Administrative boundary: FAO GAUL 2025 Level 1
admin1 = ee.FeatureCollection("FAO/GAUL/2025/level1")
thailand = admin1.filter(ee.Filter.eq("GAUL0_NAME", "Thailand"))

def get_admin1(name_contains):
    # เลือกหน่วยปกครองระดับ 1 ของประเทศไทยด้วยข้อความบางส่วนใน GAUL1_NAME
    return thailand.filter(
        ee.Filter.stringContains("GAUL1_NAME", name_contains)
    )

bangkok = get_admin1("Bangkok")
chiang_mai = get_admin1("Chiang Mai")
Kalasin= get_admin1("Kalasin")
print("Bangkok features:", bangkok.size().getInfo())
print("Chiang Mai features:", chiang_mai.size().getInfo())
print("Kalasin:", Kalasin.size().getInfo())
print("Chiang Mai features:", chiang_mai.size().getInfo())

Bangkok features: 1
Chiang Mai features: 1
Kalasin: 1
Chiang Mai features: 1


## 3. เตรียม Sentinel-2 composite

เราใช้ช่วงเวลาเดียวกับ Notebook 01 เพื่อให้การเปรียบเทียบต่อเนื่อง

In [6]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    scaled = image.select("B.*").multiply(0.0001)
    return (
        scaled.updateMask(clear)
        .copyProperties(image, ["system:time_start"])
    )

def sentinel2_composite(region, start_date, end_date, max_cloud=40):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
        .map(mask_s2_sr)
    )
    return collection, collection.median().clip(region)

START = "2024-11-01"
END   = "2025-03-01"

_, s2_bkk = sentinel2_composite('kalasin'.geometry(), START, END)

AttributeError: 'str' object has no attribute 'geometry'

## 4. คำนวณ NDVI — Bangkok

In [ ]:
ndvi_bkk = (
    s2_bkk.normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

ndvi_stats_bkk = ndvi_bkk.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bangkok.geometry(),
    scale=10,
    maxPixels=1e10,
    tileScale=4,
)

print("Mean NDVI (Bangkok):", ndvi_stats_bkk.get("NDVI").getInfo())

Mean NDVI (Bangkok): 0.38246322155564866


In [ ]:
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": ["#f7f7f7", "#d9f0a3", "#78c679", "#238443"],
}

Map_ndvi = create_course_map()
Map_ndvi.centerObject(bangkok, 9)
Map_ndvi.addLayer(ndvi_bkk, ndvi_vis, "Bangkok NDVI")
Map_ndvi.addLayer(
    bangkok.style(color="303030", fillColor="00000000", width=2),
    {},
    "Boundary"
)
Map_ndvi.add_colorbar(ndvi_vis, label="NDVI", layer_name="Bangkok NDVI")
Map_ndvi.add_layer_control()
Map_ndvi

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

## 5. Dynamic World และ Built Probability

Dataset: `GOOGLE/DYNAMICWORLD/V1`

Dynamic World ให้ probability 0–1 สำหรับหลาย class เราจะใช้ `built`

เราสร้างค่าเฉลี่ยของ `built probability` ตลอดช่วงวิเคราะห์ จากนั้นทดลอง threshold:

$$
P(\mathrm{built}) \geq 0.5
$$

**0.5 เป็น operational threshold เพื่อการสอน ไม่ใช่ค่ามาตรฐานสากล**

In [ ]:
dw_bkk = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(bangkok.geometry())
    .filterDate(START, END)
)

built_prob_bkk = dw_bkk.select("built").mean().clip(bangkok.geometry())
built_mask_bkk = built_prob_bkk.gte(0.5).selfMask().rename("built_mask")

print("Dynamic World scenes:", dw_bkk.size().getInfo())

Dynamic World scenes: 48


In [ ]:
built_vis = {
    "min": 0,
    "max": 1,
    "palette": ["#f7fbff", "#9ecae1", "#3182bd", "#08306b"],
}

Map_built = create_course_map()
Map_built.centerObject(bangkok, 9)
Map_built.addLayer(built_prob_bkk, built_vis, "Built probability")
Map_built.addLayer(built_mask_bkk, {"palette": ["#252525"]}, "Built >= 0.5", False)
Map_built.addLayer(
    bangkok.style(color="202020", fillColor="00000000", width=2),
    {},
    "Boundary"
)
Map_built.add_colorbar(built_vis, label="Built probability", layer_name="Built probability")
Map_built.add_layer_control()
Map_built

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

## 6. คำนวณ Built-up Area

Pixel area ให้หน่วยเป็น m² จึงหาร $10^6$ เพื่อแปลงเป็น km²

In [ ]:
built_area_img = ee.Image.pixelArea().updateMask(built_mask_bkk)

built_area_m2 = built_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=bangkok.geometry(),
    scale=10,
    maxPixels=1e10,
    tileScale=4,
).get("area")

total_area_m2 = bangkok.geometry().area()

built_km2 = ee.Number(built_area_m2).divide(1e6)
total_km2 = ee.Number(total_area_m2).divide(1e6)
built_pct = built_km2.divide(total_km2).multiply(100)

print(f"Total area: {total_km2.getInfo():,.1f} km²")
print(f"Estimated built area: {built_km2.getInfo():,.1f} km²")
print(f"Estimated built percentage: {built_pct.getInfo():.1f}%")

Total area: 1,577.6 km²
Estimated built area: 1,007.9 km²
Estimated built percentage: 63.9%


### อ่านแผนที่ก่อนตอบตัวเลข
- พื้นที่ probability สูงอยู่ตรงไหน?
- Threshold 0.5 ตัดพื้นที่ใดออกไป?
- Built surface มีรูปแบบต่อเนื่องหรือเป็น patch?

## 7. แบบฝึกหัด — Chiang Mai

ให้คำนวณ:
1. Mean NDVI
2. Built probability map
3. Built mask ที่ threshold 0.5
4. Total area (km²)
5. Estimated built-up area (km²)
6. Built-up percentage

**คำถาม:** ค่า built-up percentage ของ “ทั้งจังหวัดเชียงใหม่” เป็นตัวแทน urban density ของ “เมืองเชียงใหม่” ได้ดีหรือไม่?

In [ ]:
# TODO: ทำซ้ำ workflow สำหรับ chiang_mai
# cm_region = ...
# _, s2_cm = ...
# ndvi_cm = ...
# dw_cm = ...
# built_prob_cm = ...
# built_mask_cm = ...

## 8. สิ่งที่ควรได้จาก Notebook 02

- จาก “ภาพ” สู่ “ตัวแปรเชิงปริมาณ”
- NDVI เป็น spectral index ไม่ใช่ชนิด land use
- Classification probability มี uncertainty
- ผลสถิติขึ้นกับ boundary ที่เลือก

# SOLUTION — เปิดหลังทำแบบฝึกหัดแล้ว

In [ ]:
cm_region = chiang_mai.geometry()

_, s2_cm = sentinel2_composite(cm_region, START, END)
ndvi_cm = s2_cm.normalizedDifference(["B8", "B4"]).rename("NDVI")

mean_ndvi_cm = ndvi_cm.reduceRegion(
    ee.Reducer.mean(),
    cm_region,
    10,
    maxPixels=1e10,
    tileScale=4,
).get("NDVI")

dw_cm = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(cm_region)
    .filterDate(START, END)
)

built_prob_cm = dw_cm.select("built").mean().clip(cm_region)
built_mask_cm = built_prob_cm.gte(0.5).selfMask().rename("built_mask")

built_area_cm_m2 = ee.Image.pixelArea().updateMask(built_mask_cm).reduceRegion(
    ee.Reducer.sum(),
    cm_region,
    10,
    maxPixels=1e10,
    tileScale=4,
).get("area")

total_cm_km2 = ee.Number(cm_region.area()).divide(1e6)
built_cm_km2 = ee.Number(built_area_cm_m2).divide(1e6)
built_cm_pct = built_cm_km2.divide(total_cm_km2).multiply(100)

print(f"Mean NDVI: {ee.Number(mean_ndvi_cm).getInfo():.3f}")
print(f"Total area: {total_cm_km2.getInfo():,.1f} km²")
print(f"Estimated built area: {built_cm_km2.getInfo():,.1f} km²")
print(f"Estimated built percentage: {built_cm_pct.getInfo():.2f}%")

Map_cm = create_course_map()
Map_cm.centerObject(chiang_mai, 8)
Map_cm.addLayer(ndvi_cm, ndvi_vis, "Chiang Mai NDVI")
Map_cm.addLayer(built_prob_cm, built_vis, "Built probability", False)
Map_cm.addLayer(built_mask_cm, {"palette": ["#252525"]}, "Built >= 0.5", False)
Map_cm.addLayer(
    chiang_mai.style(color="202020", fillColor="00000000", width=2),
    {},
    "Boundary"
)
Map_cm.add_layer_control()
Map_cm

Mean NDVI: 0.724
Total area: 22,187.0 km²
Estimated built area: 761.1 km²
Estimated built percentage: 3.43%


Map(center=[18.78744159805189, 98.72513681282032], controls=(WidgetControl(options=['position', 'transparent_b…

### แนวคำตอบ
Built-up percentage ของทั้งจังหวัดเชียงใหม่ไม่ควรตีความตรง ๆ ว่าเป็น urban density ของตัวเมือง เพราะ denominator รวมพื้นที่ภูเขา ป่าไม้ เกษตร และชนบทจำนวนมาก นี่คือผลของ **study-area definition / scale effect** และเชื่อมโยงกับแนวคิด MAUP

## แหล่งข้อมูลอ้างอิง
- Sentinel-2 SR Harmonized: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Dynamic World V1: https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1